<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 01: Feature Backfill for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook backfills historical features for the two-stage train delay model.

It performs the following steps:

1. **Choose train stations and time range**: define a list of LocationSignature codes for the Pendeltåg network and select a backfill window (e.g., last X days/months).
2. **Fetch historical TrainAnnouncement data** from Trafikverket's Open API using XML queries.
3. **Fetch auxiliary data** such as weather observations and ReasonCodes (if available) to enrich the dataset.
4. **Engineer features** such as delay minutes, time-of-day, day-of-week, recent delays, weather metrics, and reason categories.
5. **Save the resulting DataFrame** into a Hopsworks feature group for downstream training and inference.

> 🛠️ **Note**: You need to supply a valid Trafikverket API key. The API returns JSON when the request is sent in XML format. Replace placeholders where indicated.


In [1]:
import re
import numpy as np
import pandas as pd

STOCKHOLM_TZ = "Europe/Stockholm"


### 📝 Imports

In [2]:
import os
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
import hopsworks
from typing import Any, Dict, List, Optional, Tuple


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()


True

## 📡 Connect to Hopsworks Feature Store

In [3]:
# Optional: Hopsworks storage (not required)
import hopsworks_utils
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-11 18:32:11,663 INFO: Initializing external client
2026-01-11 18:32:11,664 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-11 18:32:13,108 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2180
Hopsworks login OK


## 🔑 Configure Trafikverket API and Helper Functions

In [4]:
def normalize_lists(df: pd.DataFrame) -> pd.DataFrame:
    """Convert list-columns to comma-separated strings."""
    if df.empty:
        return df

    list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
    if list_cols:
        print(f"Flattening list columns: {list_cols}")
        for c in list_cols:
            df[c] = df[c].apply(lambda x: ",".join(map(str, x)) if isinstance(x, list) else x)
    return df


In [5]:


from zoneinfo import ZoneInfo

# ============================================================
# Trafikverket Open API (v2) helpers
# ============================================================

TRAFIKVERKET_BASE_URL = os.getenv(
    "TRAFIKVERKET_BASE_URL",
    "https://api.trafikinfo.trafikverket.se/v2/data.json",
)
API_KEY_TRAFIK = os.getenv("API_KEY_TRAFIK")

# Define Stockholm timezone for explicit conversion
STOCKHOLM_TZ = ZoneInfo("Europe/Stockholm")

def _iso(ts: dt.datetime) -> str:
    """
    Format timestamp as Trafikverket ISO string without timezone.
    Converts to Stockholm local time first, then formats as naive.
    """
    if ts.tzinfo is not None:
        ts = ts.astimezone(STOCKHOLM_TZ)
    return ts.strftime("%Y-%m-%dT%H:%M:%S")


def build_request_xml(
    api_key: str,
    object_type: str,
    filter_xml: str,
    include_fields: List[str],
    limit: int = 10000,
    schema_version: str = "1", # Weather uses v1 often, but v2 is available.
    order_by: Optional[str] = None,
) -> str:
    # 1. Format INCLUDE tags
    include_xml = "".join(f"<INCLUDE>{f}</INCLUDE>" for f in include_fields)

    # 2. Add 'orderby' as an attribute to the QUERY tag, NOT as a child element
    orderby_attr = f'orderby="{order_by}"' if order_by else ""

    return f"""
<REQUEST>
  <LOGIN authenticationkey="{api_key}" />
  <QUERY objecttype="{object_type}" schemaversion="{schema_version}" limit="{limit}" {orderby_attr}>
    {filter_xml}
    {include_xml}
  </QUERY>
</REQUEST>
""".strip()


def query_trafikverket(xml_body: str, timeout: int = 60) -> Dict[str, Any]:
    headers = {"Content-Type": "text/xml; charset=utf-8"}
    r = requests.post(
        TRAFIKVERKET_BASE_URL,
        data=xml_body.encode("utf-8"),
        headers=headers,
        timeout=timeout,
    )
    if not r.ok:
        print(f"❌ API Error {r.status_code}:")
        print(r.text)
    r.raise_for_status()
    return r.json()


def _extract_result_list(resp: Dict[str, Any], object_type: str) -> List[Dict[str, Any]]:
    """Return list of objects from RESPONSE.RESULT[0][object_type]."""
    try:
        return resp["RESPONSE"]["RESULT"][0].get(object_type, [])
    except Exception:
        return []


def _chunk_time_windows(start: dt.datetime, end: dt.datetime, hours: int) -> List[Tuple[dt.datetime, dt.datetime]]:
    out = []
    cur = start
    while cur < end:
        nxt = min(end, cur + dt.timedelta(hours=hours))
        out.append((cur, nxt))
        cur = nxt
    return out


# ============================================================
# Fetch: TrainAnnouncement (ops: timetable + actual + est + cancel)
# ============================================================

TRAINANNOUNCE_FIELDS = [
    "ActivityId",
    "ActivityType",                 # Arrival / Departure
    "AdvertisedTrainIdent",         # train number
    "AdvertisedTimeAtLocation",     # scheduled
    "EstimatedTimeAtLocation",      # predicted
    "TimeAtLocation",               # actual (when available)
    "LocationSignature",            # station code
    "Canceled",
    "Deleted",
    "InformationOwner",
    "Deviation",                    # sometimes contains cause info
    "FromLocation",
    "ToLocation",
    "TrackAtLocation",
]


def fetch_train_announcements(
    station_codes: List[str],
    start_time: dt.datetime,
    end_time: dt.datetime,
    window_hours: int = 6,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    Backfill TrainAnnouncement data by splitting time into windows.
    """
    all_rows: List[Dict[str, Any]] = []
    windows = _chunk_time_windows(start_time, end_time, hours=window_hours)

    # Indentation for XML readability
    station_or = "\n      ".join([f'<EQ name="LocationSignature" value="{s}" />' for s in station_codes])

    for (ws, we) in windows:
        filter_xml = f"""
<FILTER>
  <AND>
    <GT name="AdvertisedTimeAtLocation" value="{_iso(ws)}" />
    <LT name="AdvertisedTimeAtLocation" value="{_iso(we)}" />
    <OR>
      {station_or}
    </OR>
  </AND>
</FILTER>
""".strip()

        xml = build_request_xml(
            api_key=api_key,
            object_type="TrainAnnouncement",
            filter_xml=filter_xml,
            include_fields=TRAINANNOUNCE_FIELDS,
            limit=limit,
            order_by="AdvertisedTimeAtLocation",
        )
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "TrainAnnouncement")
        all_rows.extend(rows)

    if not all_rows:
        return pd.DataFrame()

    df = pd.json_normalize(all_rows)
    df = normalize_lists(df)
    
    

    
    # parse timestamps (some may be missing)
    for col in ["AdvertisedTimeAtLocation", "EstimatedTimeAtLocation", "TimeAtLocation"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # standardize names
    df = df.rename(
        columns={
            "AdvertisedTimeAtLocation": "scheduled_time",
            "EstimatedTimeAtLocation": "estimated_time",
            "TimeAtLocation": "actual_time",
            "LocationSignature": "station_code",
            "AdvertisedTrainIdent": "train_id",
        }
    )

    # choose best available "observed" time for historical delay calc
    df["observed_time"] = df["actual_time"].fillna(df["estimated_time"])

    # delay in minutes
    df["delay_min"] = (df["observed_time"] - df["scheduled_time"]).dt.total_seconds() / 60.0

    # event_time = scheduled_time for point-in-time alignment
    df["event_time"] = df["scheduled_time"]

    # cancellation flag normalize
    df["is_canceled"] = df.get("Canceled", False).fillna(False).astype(bool)

    # basic calendar fields
    df["hour"] = df["event_time"].dt.hour
    df["dow"] = df["event_time"].dt.dayofweek
    df["date"] = df["event_time"].dt.date

    # keep only the core columns we need downstream
    keep = [
        "ActivityId",
        "ActivityType",
        "train_id",
        "OperationalTrainNumber",
        "event_time",
        "scheduled_time",
        "estimated_time",
        "actual_time",
        "observed_time",
        "station_code",
        "delay_min",
        "is_canceled",
        "Deleted",
        "InformationOwner",
        "Deviation",
        "FromLocation",
        "ToLocation",
        "TrackAtLocation",
        "hour",
        "dow",
        "date",
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].sort_values(["train_id", "event_time", "station_code"]).reset_index(drop=True)
    return df


# ============================================================
# Fetch: TrainMessage (incident-ish, optional)
# ============================================================

TRAINMESSAGE_FIELDS = [
    "ExternalDescription",
    "ReasonCodeText",
    "StartDateTime",
    "LastUpdateDateTime",
    "AffectedLocation",
    "EventId",
]


def fetch_train_messages(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """Fetch TrainMessage in a time window."""

    # Simple filter on StartDateTime only
    filter_xml = f"""
<FILTER>
  <AND>
    <LT name="StartDateTime" value="{_iso(end_time)}" />
    <GT name="StartDateTime" value="{_iso(start_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="TrainMessage",
        filter_xml=filter_xml,
        include_fields=TRAINMESSAGE_FIELDS,
        limit=limit,
        order_by="StartDateTime",
    )
    resp = query_trafikverket(xml)
    rows = _extract_result_list(resp, "TrainMessage")
    if not rows:
        return pd.DataFrame()

    df = pd.json_normalize(rows)

    # Parse dates
    for col in ["StartDateTime", "LastUpdateDateTime", "CreatedDateTime", "LastModifiedDateTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================



# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================

# CORRECTED FIELDS: 'MeasurementTime' is the correct field for time.
# We use Schema v2 notation (dot notation) for Temperature/Wind.
WEATHER_OBS_FIELDS = [
    "MeasurementTime",      # Fixed: was ObservationTime
    "ModifiedTime",
    "Id",
    "Air.Temperature",
    "Wind.Speed",
]

def fetch_weather_observations_last7d(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    WeatherObservation only keeps ~7 days historically.
    Returns empty if outside retention window.
    """
    filter_xml = f"""
<FILTER>
  <AND>
    <GT name="MeasurementTime" value="{_iso(start_time)}" />
    <LT name="MeasurementTime" value="{_iso(end_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="WeatherObservation",
        filter_xml=filter_xml,
        include_fields=WEATHER_OBS_FIELDS,
        limit=limit,
        order_by="MeasurementTime",
    )

    # We anticipate this might return empty for old dates, so we handle it gracefully.
    try:
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "WeatherObservation")
    except Exception as e:
        # If it fails (e.g. timeout or syntax), print but don't crash the whole pipeline
        print(f"⚠️ Weather fetch warning: {e}")
        return pd.DataFrame()

    if not rows:
        # Expected for 2023 dates (data purged after 7 days)
        return pd.DataFrame()

    df = pd.json_normalize(rows)
    for col in ["MeasurementTime", "ModifiedTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df = df.rename(columns={"MeasurementTime": "weather_time"})
    return df

# ============================================================
# Join logic (ops ↔ incidents) + label creation
# ============================================================
def join_train_messages(
    ops_df: pd.DataFrame,
    msg_df: pd.DataFrame,
    time_buffer_min: int = 15,
) -> pd.DataFrame:
    """
    Best-effort incident join.

    We join TrainMessage onto stop events by station_code and time overlap.
    Since TrainMessage has no EndTime in the API, we assume it applies for 60 mins.

    Key improvement vs earlier version:
    - After merging, we deduplicate to ONE row per (train_id, event_time, station_code),
      keeping the latest *stop update* (ModifiedTime) and then the latest message (StartDateTime).
    """
    if ops_df.empty or msg_df.empty:
        ops_df = ops_df.copy()
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        ops_df["reason_desc"] = None
        return ops_df

    m = msg_df.copy()

    # Flatten AffectedLocation
    affected_col = None
    for cand in ["AffectedLocation", "AffectedLocation.LocationSignature"]:
        if cand in m.columns:
            affected_col = cand
            break

    if affected_col is None:
        ops_df = ops_df.copy()
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        ops_df["reason_desc"] = None
        return ops_df

    if affected_col == "AffectedLocation":
        m = m.explode("AffectedLocation")
        m["affected_station"] = m["AffectedLocation"].apply(
            lambda x: x.get("LocationSignature") if isinstance(x, dict) else None
        )
    else:
        m["affected_station"] = m[affected_col]

    m = m.dropna(subset=["affected_station"]).copy()

    # Create synthetic end time (Start + 1 hour)
    if "StartDateTime" in m.columns:
        m["StartDateTime"] = pd.to_datetime(m["StartDateTime"], errors="coerce")
    m["effective_end"] = m["StartDateTime"] + pd.Timedelta(hours=1)

    ops = ops_df.copy()

    # Ensure event_time is datetime
    ops["event_time"] = pd.to_datetime(ops["event_time"], errors="coerce")

    ops["t_start"] = ops["event_time"] - pd.to_timedelta(time_buffer_min, unit="m")
    ops["t_end"] = ops["event_time"] + pd.to_timedelta(time_buffer_min, unit="m")

    # Select columns to merge (only those that exist)
    cols_to_use = ["StartDateTime", "effective_end", "affected_station"]
    for c in ["ReasonCodeText", "ExternalDescription"]:
        if c in m.columns:
            cols_to_use.append(c)
    if "ReasonCode" in m.columns:
        cols_to_use.append("ReasonCode")

    merged = ops.merge(
        m[cols_to_use],
        left_on="station_code",
        right_on="affected_station",
        how="left",
    )

    # Keep rows with no message OR overlapping message window
    overlap = (
        merged["StartDateTime"].isna()
        | ((merged["StartDateTime"] <= merged["t_end"]) & (merged["effective_end"] >= merged["t_start"]))
    )
    merged = merged[overlap].copy()

    # ----------------------------
    # DEDUP (IMPROVED)
    # One row per stop event:
    #   (train_id, event_time, station_code)
    # Keep the latest stop update first (ModifiedTime),
    # then the latest message time (StartDateTime).
    # ----------------------------
    merged["msg_rank_time"] = merged["StartDateTime"].fillna(pd.Timestamp.min)

    if "ModifiedTime" in merged.columns:
        merged["ModifiedTime"] = pd.to_datetime(merged["ModifiedTime"], errors="coerce")
        merged["stop_rank_time"] = merged["ModifiedTime"].fillna(pd.Timestamp.min)
        merged = merged.sort_values(
            ["train_id", "event_time", "stop_rank_time", "msg_rank_time"],
            ascending=[True, True, False, False],
        )
    else:
        merged = merged.sort_values(
            ["train_id", "event_time", "msg_rank_time"],
            ascending=[True, True, False],
        )

    merged = merged.drop_duplicates(
        subset=["train_id", "event_time", "station_code"],
        keep="first",
    )

    # Rename / clean up
    merged = merged.rename(columns={
        "ReasonCodeText": "reason_text",
        "ExternalDescription": "reason_desc",
        "ReasonCode": "reason_code",
    })

    drop_cols = ["t_start", "t_end", "affected_station", "msg_rank_time", "effective_end", "stop_rank_time"]
    merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns], errors="ignore")

    # Ensure columns exist even if message fields missing
    for col in ["reason_code", "reason_text", "reason_desc"]:
        if col not in merged.columns:
            merged[col] = None

    return merged


def add_labels(
    df: pd.DataFrame,
    horizon_min: int = 60,
    delay_threshold_min: int = 10,
) -> pd.DataFrame:
    """
    Create:
    - predictive label: will delay exceed threshold within next horizon?
    - reactive labels: final_delay for that train/day, additional_delay from now
    """
    if df.empty:
        return df

    out = df.copy()
    out["train_run_id"] = out["train_id"].astype(str) + "_" + out["date"].astype(str)

    # final delay per train run: use last known delay
    out["final_delay_min"] = out.groupby("train_run_id")["delay_min"].transform("last")
    out["additional_delay_min"] = out["final_delay_min"] - out["delay_min"]

    out = out.sort_values(["train_run_id", "event_time"]).reset_index(drop=True)
    horizon = pd.Timedelta(minutes=horizon_min)

    pred_flags = []
    for _, g in out.groupby("train_run_id", sort=False):
        times = g["event_time"].to_numpy()
        delays = g["delay_min"].to_numpy()
        y = []
        for i in range(len(g)):
            t0 = times[i]
            j = i
            max_d = -1e9
            while j < len(g) and (times[j] - t0) <= horizon:
                if pd.notna(delays[j]):
                    if delays[j] > max_d:
                        max_d = delays[j]
                j += 1
            y.append(1 if max_d >= delay_threshold_min else 0)
        pred_flags.extend(y)

    out["y_delay_within_horizon"] = pred_flags
    return out

In [6]:
import re
import numpy as np
# ============================================================
# Fetch: TrainStation (static metadata)
# ============================================================

STATION_FIELDS = [
    "LocationSignature",
    "AdvertisedLocationName",
    "Geometry.WGS84",
]

def fetch_all_stations(api_key: str = API_KEY_TRAFIK) -> pd.DataFrame:
    """Fetch all train stations with geometry."""
    # We fetch ALL stations (empty filter) to ensure we get the coordinates
    xml = build_request_xml(
        api_key=api_key,
        object_type="TrainStation",
        filter_xml="", 
        include_fields=STATION_FIELDS,
    )
    
    resp = query_trafikverket(xml)
    rows = _extract_result_list(resp, "TrainStation")
    
    if not rows:
        return pd.DataFrame()
        
    df = pd.json_normalize(rows)
    return df

print("Fetching stations...")
stations_df = fetch_all_stations()
print(f"✅ Loaded {len(stations_df)} stations.")
def parse_wgs84_point(s):
    """
    Expects: "POINT (lon lat)"  e.g. "POINT (18.0687 59.3294)"
    """
    if not isinstance(s, str):
        return (np.nan, np.nan)
    m = re.search(r"POINT\s*\(\s*([0-9.\-]+)\s+([0-9.\-]+)\s*\)", s)
    if not m:
        return (np.nan, np.nan)
    lon = float(m.group(1))
    lat = float(m.group(2))
    return (lat, lon)

geom_col = "Geometry.WGS84"
if geom_col not in stations_df.columns:
    raise KeyError(f"'{geom_col}' not found in stations_df columns: {list(stations_df.columns)}")

stations_geo = stations_df.copy()
stations_geo[["lat", "lon"]] = stations_geo[geom_col].apply(lambda x: pd.Series(parse_wgs84_point(x)))
stations_geo = stations_geo.dropna(subset=["lat", "lon"])

stations_geo = stations_geo.rename(columns={"Geometry.WGS84": "wgs84"})
display(stations_geo[["AdvertisedLocationName","LocationSignature","lat","lon"]].head(10))


Fetching stations...
✅ Loaded 1745 stations.


,AdvertisedLocationName,LocationSignature,lat,lon
0,Alingsås,A,57.926905,12.532185
1,Anneberg,Ag,57.538629,12.100776
2,Aneby,Any,57.837435,14.811896
3,Aspen,Apn,57.754422,12.240512
4,Arvika,Ar,59.653634,12.590803
5,Arboga,Arb,59.397189,15.840526
6,Arlanda C,Arnc,59.649072,17.928489
7,Aspedalen,Asd,57.762478,12.258478
8,Avesta Krylbo,Avky,60.129533,16.216148
9,Barkåkra,Baa,56.293656,12.824632


## 🕒 Define Backfill Parameters

In [7]:
# Stockholm län bbox
MIN_LON, MIN_LAT = 17.25, 58.69
MAX_LON, MAX_LAT = 19.61, 60.27

stockholm_lan_df = stations_geo[
    (stations_geo["lon"] >= MIN_LON) & (stations_geo["lon"] <= MAX_LON) &
    (stations_geo["lat"] >= MIN_LAT) & (stations_geo["lat"] <= MAX_LAT)
].copy()

print("Stations in Stockholm län bbox:", len(stockholm_lan_df))

station_codes_stockholm_lan = (
    stockholm_lan_df["LocationSignature"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

station_codes_stockholm_lan[:30], len(station_codes_stockholm_lan)
station_codes = station_codes_stockholm_lan

Stations in Stockholm län bbox: 210


In [8]:
# Increase time window for more trains
end_time = dt.datetime.now(dt.timezone.utc) + dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=30)

print("Start Time: ", start_time)

print("End Time: ", end_time)


#deth = 1/0
ops_df = fetch_train_announcements(
    station_codes=station_codes,
    start_time=start_time,
    end_time=end_time,
    window_hours=3,
    limit=50000,
)
print("RAW ops_df:", ops_df.shape)
print("Unique stop events (train_id,event_time,station_code):",
      ops_df.drop_duplicates(["train_id","event_time","station_code"]).shape)


print("ops rows:", len(ops_df))
display(ops_df.head())


Start Time:  2025-12-13 17:32:14.113802+00:00
End Time:  2026-01-12 17:32:14.113802+00:00
Flattening list columns: ['Deviation', 'FromLocation', 'ToLocation']
RAW ops_df: (78539, 20)
Unique stop events (train_id,event_time,station_code): (47898, 20)
ops rows: 78539


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,is_canceled,Deleted,InformationOwner,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date
0,1500adde-075d-66fb-08de-4372aca74cdc,Avgang,1,2026-01-09 23:17:00+01:00,2026-01-09 23:17:00+01:00,2026-01-10 00:15:00+01:00,2026-01-10 00:30:00+01:00,2026-01-10 00:30:00+01:00,Cst,73.0,False,False,SJ,Inväntat tåg,Cst,"Nr,Lp,Hm,Hb,Mc",12,23,4,2026-01-09
1,1500adde-075d-66fb-08de-4506ed9a6b79,Avgang,1,2026-01-11 23:17:00+01:00,2026-01-11 23:17:00+01:00,NaT,NaT,NaT,Cst,NaN,False,False,SJ,NaN,Cst,"Nr,Lp,Hm,Hb,Mc",12,23,6,2026-01-11
2,1500adde-075d-66fb-08de-4372acb23403,Avgang,10,2026-01-09 12:11:00+01:00,2026-01-09 12:11:00+01:00,NaT,2026-01-09 12:14:00+01:00,2026-01-09 12:14:00+01:00,Cst,3.0,False,False,SJ,NaN,Cst,"U,Gä,Suc,Ös,Du",10,12,4,2026-01-09
3,1500adde-075d-66fb-08de-4372acb23404,Ankomst,10,2026-01-09 12:33:00+01:00,2026-01-09 12:33:00+01:00,2026-01-09 12:37:00+01:00,2026-01-09 12:37:00+01:00,2026-01-09 12:37:00+01:00,Arnc,4.0,False,False,SJ,NaN,Cst,Du,1,12,4,2026-01-09
4,1500adde-075d-66fb-08de-4372acb23405,Avgang,10,2026-01-09 12:34:00+01:00,2026-01-09 12:34:00+01:00,2026-01-09 12:37:00+01:00,2026-01-09 12:40:00+01:00,2026-01-09 12:40:00+01:00,Arnc,6.0,False,False,SJ,NaN,Cst,"U,Gä,Suc,Ös,Du",1,12,4,2026-01-09


In [9]:
#Sort ops_df by event_time descending and print the first few rows
ops_df = ops_df.sort_values(by="event_time", ascending=False)

display(ops_df.head())

,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,is_canceled,Deleted,InformationOwner,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date
46934,1500adde-075d-66fb-08de-45d1ccf94f06,Ankomst,2659,2026-01-12 18:32:00+01:00,2026-01-12 18:32:00+01:00,NaT,NaT,NaT,Hel,NaN,False,False,SL,NaN,Mr,Söc,3,18,0,2026-01-12
47101,1500adde-075d-66fb-08de-45d1cd2027ca,Avgang,2660,2026-01-12 18:32:00+01:00,2026-01-12 18:32:00+01:00,NaT,NaT,NaT,Tu,NaN,False,False,SL,NaN,Söc,"Sci,Mr",4,18,0,2026-01-12
14108,1500adde-075d-66fb-08de-45d1b042bd89,Avgang,2256,2026-01-12 18:32:00+01:00,2026-01-12 18:32:00+01:00,NaT,NaT,NaT,R,NaN,False,False,SL,NaN,Söc,"Arnc,U",2,18,0,2026-01-12
14109,1500adde-075d-66fb-08de-45d1b042bd88,Ankomst,2256,2026-01-12 18:32:00+01:00,2026-01-12 18:32:00+01:00,NaT,NaT,NaT,R,NaN,False,False,SL,NaN,Söc,U,2,18,0,2026-01-12
46267,1500adde-075d-66fb-08de-45d1cc7b8fa6,Ankomst,2655,2026-01-12 18:32:00+01:00,2026-01-12 18:32:00+01:00,NaT,NaT,NaT,Söc,NaN,False,False,SL,NaN,Mr,Söc,2,18,0,2026-01-12


In [10]:
#msg_df = fetch_train_messages_new(start_time=start_time, end_time=end_time)
#print("messages rows:", len(msg_df))
#display(msg_df.head())


In [11]:
#ops_df = join_train_messages(ops_df, msg_df)
#print("ops rows after joining messages:", len(ops_df))
#display(ops_df.head())


In [12]:
import requests
import pandas as pd
from datetime import timedelta
from zoneinfo import ZoneInfo

HOURLY_VARS = "temperature_2m,precipitation,rain,snowfall,windspeed_10m"
TZ = "Europe/Stockholm"

def _to_hourly_df(j: dict) -> pd.DataFrame:
    if "hourly" not in j or "time" not in j["hourly"]:
        return pd.DataFrame()
    h = pd.DataFrame(j["hourly"])
    h["weather_time"] = pd.to_datetime(h["time"])
    h = h.drop(columns=["time"])
    return h

def fetch_openmeteo_archive(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": HOURLY_VARS,
        "timezone": TZ,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    h = _to_hourly_df(r.json())
    if not h.empty:
        h["source"] = "archive"
    return h

def fetch_openmeteo_forecast(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": HOURLY_VARS,
        "timezone": TZ,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    h = _to_hourly_df(r.json())
    if not h.empty:
        h["source"] = "forecast"
    return h

def fetch_weather_around_today(
    lat: float,
    lon: float,
    past_days: int = 7,
    future_days: int = 3,
) -> pd.DataFrame:
    """
    Fetch hourly weather for:
      - past_days back from today (inclusive range ends yesterday)
      - future_days ahead from today (inclusive range ends future_days days after today)

    Returns one combined DataFrame:
      [temperature_2m, precipitation, rain, snowfall, windspeed_10m, weather_time, source]
    with timezone-aware weather_time in Europe/Stockholm.
    """
    today = pd.Timestamp.now(tz=ZoneInfo(TZ)).date()

    past_start = today - timedelta(days=past_days)
    past_end = today - timedelta(days=1)      # yesterday
    future_start = today                      # today
    future_end = today + timedelta(days=future_days)

    dfs = []

    # Past via archive
    if past_days > 0:
        dfs.append(
            fetch_openmeteo_archive(
                lat, lon,
                start_date=past_start.isoformat(),
                end_date=past_end.isoformat()
            )
        )

    # Future via forecast (includes today..future_end)
    if future_days >= 0:
        dfs.append(
            fetch_openmeteo_forecast(
                lat, lon,
                start_date=future_start.isoformat(),
                end_date=future_end.isoformat()
            )
        )

    out = pd.concat([d for d in dfs if d is not None and not d.empty], ignore_index=True)
    if out.empty:
        return out

    # Make weather_time tz-aware (Open-Meteo returns local times; pandas parses as naive)
    out["weather_time"] = (
        pd.to_datetime(out["weather_time"])
          .dt.tz_localize(TZ, nonexistent="shift_forward", ambiguous="NaT")
    )

    # Sort + dedupe, prefer archive over forecast if overlap exists
    out["source_rank"] = out["source"].map({"archive": 0, "forecast": 1}).fillna(99).astype(int)
    out = (
        out.sort_values(["weather_time", "source_rank"])
           .drop_duplicates(subset=["weather_time"], keep="first")
           .drop(columns=["source_rank"])
           .reset_index(drop=True)
    )

    return out

# Example: Stockholm
weather_hourly = fetch_weather_around_today(
    lat=59.3293, lon=18.0686,
    past_days=7,
    future_days=3
)

print(len(weather_hourly))
print(weather_hourly.head())
print(weather_hourly.tail())


264
   temperature_2m  precipitation  rain  snowfall  windspeed_10m  \
0            -2.0            0.0   0.0       0.0           20.8   
1            -2.5            0.0   0.0       0.0           20.1   
2            -2.8            0.0   0.0       0.0           19.7   
3            -3.2            0.0   0.0       0.0           20.5   
4            -3.7            0.0   0.0       0.0           20.7   

               weather_time   source  
0 2026-01-04 00:00:00+01:00  archive  
1 2026-01-04 01:00:00+01:00  archive  
2 2026-01-04 02:00:00+01:00  archive  
3 2026-01-04 03:00:00+01:00  archive  
4 2026-01-04 04:00:00+01:00  archive  
     temperature_2m  precipitation  rain  snowfall  windspeed_10m  \
259            -0.8            0.0   0.0       0.0           21.7   
260            -0.9            0.0   0.0       0.0           21.4   
261            -0.9            0.0   0.0       0.0           21.2   
262            -1.0            0.0   0.0       0.0           21.2   
263           

In [13]:
# --- Labels (CREATE df) ---
df = add_labels(
    ops_df,
    horizon_min=1440,
    delay_threshold_min=10
)

print("df rows:", len(df))
#display(df.head())


df rows: 78539


In [14]:
import pandas as pd

# 1) Copy + parse datetimes
df2 = df.copy()
w2  = weather_hourly.copy()

df2["event_time"]   = pd.to_datetime(df2["event_time"], errors="coerce")
w2["weather_time"]  = pd.to_datetime(w2["weather_time"], errors="coerce")

# 2) Make BOTH timezone-aware in Europe/Stockholm
#    - If df event_time is naive: localize to Stockholm
#    - If df event_time is tz-aware: convert to Stockholm
if df2["event_time"].dt.tz is None:
    df2["event_time"] = df2["event_time"].dt.tz_localize(
        "Europe/Stockholm", nonexistent="shift_forward", ambiguous="NaT"
    )
else:
    df2["event_time"] = df2["event_time"].dt.tz_convert("Europe/Stockholm")

# weather_time from our new fetcher should already be tz-aware in Stockholm,
# but this makes it robust if it isn't for some reason.
if w2["weather_time"].dt.tz is None:
    w2["weather_time"] = w2["weather_time"].dt.tz_localize(
        "Europe/Stockholm", nonexistent="shift_forward", ambiguous="NaT"
    )
else:
    w2["weather_time"] = w2["weather_time"].dt.tz_convert("Europe/Stockholm")

# 3) Drop invalid timestamps
df2 = df2.dropna(subset=["event_time"]).copy()
w2  = w2.dropna(subset=["weather_time"]).copy()

# 4) Sort (required for merge_asof)
df2 = df2.sort_values("event_time").reset_index(drop=True)
w2  = w2.sort_values("weather_time").reset_index(drop=True)

print(df2["event_time"].dtype, w2["weather_time"].dtype)
print("df2 rows:", len(df2), "w2 rows:", len(w2))
print("event_time range:", df2["event_time"].min(), "->", df2["event_time"].max())
print("weather_time range:", w2["weather_time"].min(), "->", w2["weather_time"].max())

# 5) Nearest-hour merge (tolerance 1 hour)
df2 = pd.merge_asof(
    df2,
    w2,
    left_on="event_time",
    right_on="weather_time",
    direction="backward",
    tolerance=pd.Timedelta("1H"),
)

display(df2.head())
df2.sort_values("event_time").head(10)
df = df2


datetime64[ns, Europe/Stockholm] datetime64[ns, Europe/Stockholm]
df2 rows: 78539 w2 rows: 264
event_time range: 2026-01-09 00:00:00+01:00 -> 2026-01-12 18:32:00+01:00
weather_time range: 2026-01-04 00:00:00+01:00 -> 2026-01-14 23:00:00+01:00


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time,source
0,1500adde-075d-66fb-08de-437328465cb5,Avgang,2983,2026-01-09 00:00:00+01:00,2026-01-09 00:00:00+01:00,NaT,2026-01-09 00:00:00+01:00,2026-01-09 00:00:00+01:00,Mr,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
1,1500adde-075d-66fb-08de-4373071fa14a,Avgang,2477,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,NaT,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,Söc,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
2,1500adde-075d-66fb-08de-437328465cb6,Ankomst,2983,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,NaT,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,Rs,-1.0,...,0.0,1.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
3,1500adde-075d-66fb-08de-437328465cb7,Avgang,2983,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,NaT,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,Rs,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
4,1500adde-075d-66fb-08de-4373537017ba,Avgang,7878,2026-01-09 00:05:00+01:00,2026-01-09 00:05:00+01:00,NaT,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,Cst,-1.0,...,-1.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive


## 🧬 Create Feature Group and Insert Historical Data

In [15]:
df.sort_values(by=["event_time"], ascending=True, inplace=True)
display(df.head())

,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time,source
0,1500adde-075d-66fb-08de-437328465cb5,Avgang,2983,2026-01-09 00:00:00+01:00,2026-01-09 00:00:00+01:00,NaT,2026-01-09 00:00:00+01:00,2026-01-09 00:00:00+01:00,Mr,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
1,1500adde-075d-66fb-08de-4373071fa14a,Avgang,2477,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,NaT,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,Söc,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
2,1500adde-075d-66fb-08de-437328465cb6,Ankomst,2983,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,NaT,2026-01-09 00:03:00+01:00,2026-01-09 00:03:00+01:00,Rs,-1.0,...,0.0,1.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
3,1500adde-075d-66fb-08de-437328465cb7,Avgang,2983,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,NaT,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,Rs,0.0,...,0.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive
4,1500adde-075d-66fb-08de-4373537017ba,Avgang,7878,2026-01-09 00:05:00+01:00,2026-01-09 00:05:00+01:00,NaT,2026-01-09 00:04:00+01:00,2026-01-09 00:04:00+01:00,Cst,-1.0,...,-1.0,0.0,0,-2.0,0.0,0.0,0.0,8.4,2026-01-09 00:00:00+01:00,archive


In [17]:
import os, datetime as dt

# Optional: Hopsworks storage (not required)
import hopsworks_utils
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed.")
    print("Reason:", repr(e))


out_path = "data/train_stop_events_labeled.parquet"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

if "df" not in globals() or df is None or df.empty:
    print("No labeled data produced; nothing to save.")
else:
    project.push(df, "train_stop_events_labeled", ["ActivityId"], desc="Train stop events with delay labels and weather features.")
    project.push(stations_geo, "station_features", ["LocationSignature"], desc="Station features.")


    print("✅ Saved canonical labeled table:", out_path)


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-11 18:34:55,833 INFO: Closing external client and cleaning up certificates.
2026-01-11 18:34:55,837 INFO: Connection closed.
2026-01-11 18:34:55,842 INFO: Initializing external client
2026-01-11 18:34:55,843 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-11 18:34:57,223 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2180
Hopsworks login OK
2026-01-11 18:35:19,900 INFO: Computing insert statistics
Inserted historical data into feature group "train_delay_features"
2026-01-11 18:35:37,114 INFO: Computing insert statistics
Inserted historical data into feature group "train_delay_features"
✅ Saved canonical labeled table: data/train_stop_events_labeled.parquet


In [ ]:

# --- Additional feature engineering and baseline creation ---
# Import helper functions from features.py
from features import ensure_event_time, add_cause_flags, add_station_congestion_features, build_duration_baseline

# Ensure event_time is timezone‑aware (Europe/Stockholm)
try:
    df = ensure_event_time(df)
except Exception:
    # If df is not yet defined, skip
    pass

# Add cause flags based on reason text (if reason fields exist)
try:
    df = add_cause_flags(df)
except Exception:
    pass

# Add station congestion features based on historical delay patterns
try:
    df = add_station_congestion_features(df)
except Exception:
    pass

# Build and save a duration baseline (median additional delay per station/cause/hour)
try:
    duration_baseline = build_duration_baseline(df)
    baseline_path = 'data/feature_pipeline_outputs/duration_baseline.parquet'
    import os
    os.makedirs(os.path.dirname(baseline_path), exist_ok=True)
    if not duration_baseline.empty:
        duration_baseline.to_parquet(baseline_path, index=False)
        print(f"Saved duration baseline to {baseline_path} (rows: {len(duration_baseline)})")
except Exception as e:
    print(f"Could not build/save duration baseline: {e}")


Saved duration baseline to data/feature_pipeline_outputs/duration_baseline.parquet (rows: 2234)


In [ ]:
print("done")

done
